# Choosing the number of factors — explained variance

Fits the ICA for every number of components from 2 to 100 and scores how much
of the coefficient matrix each reconstruction explains, against the 15 the
analysis settles on.

`ICtest` is deliberately not loaded here; see the warning in `01`.

## Setup

In [ ]:
library(ica)        # icaimax()
library(ggplot2)
library(reshape2)   # melt()

source("../paths.R")   # SCREEN_DATA, REFERENCE, TEXTFILES; see data/README.md

selCoefs <- read.csv(referencePath("SignificantCoefMatrix.csv"), row.names = 1)
selCoefs <- t(selCoefs)
colnames(selCoefs) <- sapply(colnames(selCoefs), function(x) strsplit(x, "_")[[1]][2])
dim(selCoefs)

## Explained variance against the number of components

In [ ]:
myR2 <- function(preds, actual) {
    rss <- sum((preds - actual)^2)                  # residual sum of squares
    tss <- sum((actual - mean(actual))^2)           # total sum of squares
    1 - rss / tss
}

lappend <- function(lst, obj) { lst[[length(lst) + 1]] <- obj; lst }

actualVals <- melt(selCoefs)$value
allR2 <- list()

for (i in 2:100) {
    k <- icaimax(selCoefs, nc = i, center = TRUE)
    reconstVal <- melt(k$S %*% t(k$M))$value
    allR2 <- lappend(allR2, c("k" = i, "R2" = myR2(reconstVal, actualVals)))
}

allR2 <- data.frame(do.call(rbind, allR2))
write.csv(allR2, "outputs/ExplainedVarianceByNumberOfFactors.csv", row.names = FALSE)
tail(allR2, 3)

## The curve

The red line marks the 15 components the analysis uses.

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 6)

p <- ggplot(data = allR2, aes(x = k, y = R2)) +
    geom_line() +
    geom_point() +
    geom_vline(xintercept = 15, color = "red") +
    theme_minimal() +
    xlab("Number of components") +
    ylab("Explained variance in beta matrix") +
    scale_x_continuous(breaks = seq(0, 100, 15)) +
    scale_y_continuous(breaks = seq(0, 0.8, 0.1)) +
    theme(axis.text = element_text(size = 15), axis.title = element_text(size = 20))

pdf("figures/ExplainedVarianceByNumberOfFactors.pdf", width = 6, height = 6)
print(p)
dev.off()

p   # show it in the notebook as well as saving it